# SIH26006 — Phase B: Asset/Horizon Behavior → Δfreight Forecasting

**B3 is FROZEN until B2 results confirm Δ wins. `decision_engine.py` is NOT touched.**

## Pipeline
```
B1 → Viability map (diagnostic, not a gate)
B2 → Level-Ridge vs Δ-Ridge on locked test set (ALL 20 pairs)
  → Δ wins majority? → B3 (integration) — separate session
  → Δ loses?         → kill hypothesis, attack features/model
```

## Key design decisions (pre-registered)
- **Viability** uses ONLY `outputs/phase8_final_benchmark.csv` (clean locked level benchmark)
- **GRU/LSTM results** are diagnostic only — not used to gate B2 testing
- **K sweep** = {10, 20, 30, 50} — locked on VAL Macro-F1@2%, NOT test
- **Confidence** = validation residual P10/P50/P90 distribution, NOT `1 - sMAPE/100`
- **Win condition** (strong bar): Δ beats strongest naive on locked test AND improves sMAPE + DirAcc vs persistence
- **Isolation contract**: No existing `outputs/` or `src/` files modified

In [ ]:
# ================================================================================
# CELL 1: REPOSITORY SETUP
# ================================================================================
import os
import numpy as np
import pandas as pd

np.random.seed(42)

if not os.path.exists('outputs/modeling_dataset.csv'):
    if os.path.exists('FICOS-Platform/outputs/modeling_dataset.csv'):
        %cd FICOS-Platform
    else:
        !git clone https://github.com/SSOHEB/FICOS-Platform.git
        %cd FICOS-Platform

# Always pull latest
!git pull origin main

os.makedirs('outputs/b1_viability',  exist_ok=True)
os.makedirs('outputs/delta_forecast', exist_ok=True)

print(f'CWD: {os.getcwd()}')
print(f'Dataset   : {os.path.exists("outputs/modeling_dataset.csv")}')
print(f'Benchmark : {os.path.exists("outputs/phase8_final_benchmark.csv")}')
print(f'B1 script : {os.path.exists("scratch/b1_viability_map.py")}')
print(f'B2 script : {os.path.exists("scratch/b2_delta_forecast.py")}')

In [ ]:
# ================================================================================
# CELL 2: B1 — ASSET/HORIZON VIABILITY MAP
# ================================================================================
# Runtime: ~30 seconds
# Reads: outputs/phase8_final_benchmark.csv (clean locked level benchmark ONLY)
# Outputs: outputs/b1_viability/b1_viability_map.csv
#          outputs/b1_viability/b1_volatility_diagnosis.csv
!python scratch/b1_viability_map.py

In [ ]:
# ================================================================================
# CELL 3: B2 — LEVEL vs DELTA FORECASTING (ALL 20 PAIRS)
# ================================================================================
# Runtime: 8-20 min on Colab CPU
# K sweep {10,20,30,50} tuned on VAL Macro-F1@2%
# Alpha {1,10,100,1000,10000} tuned on VAL
# Test evaluated ONCE per configuration
# Outputs: outputs/delta_forecast/b2_comparison_scoreboard.csv
#          outputs/delta_forecast/b2_test_predictions.csv
!python scratch/b2_delta_forecast.py

In [ ]:
# ================================================================================
# CELL 4: DISPLAY B1 — VIABILITY MAP
# ================================================================================
import pandas as pd

if os.path.exists('outputs/b1_viability/b1_viability_map.csv'):
    df_b1 = pd.read_csv('outputs/b1_viability/b1_viability_map.csv')
    print('=' * 70)
    print('B1 — VIABILITY MAP  (C1=R²>0, C2=sMAPE<20%, C3=beats persistence MAE)')
    print('=' * 70)
    display(df_b1[['asset','horizon','test_R2','test_sMAPE','test_MAE',
                   'persistence_MAE','C1_R2_pos','C2_sMAPE_lt20',
                   'C3_beats_persistence','conditions_passed',
                   'viability_tier','delta_test_priority']])

    print('\nVolatility diagnosis:')
    df_atr = pd.read_csv('outputs/b1_viability/b1_volatility_diagnosis.csv')
    display(df_atr)
else:
    print('Run Cell 2 first.')

In [ ]:
# ================================================================================
# CELL 5: DISPLAY B2 — FULL SCOREBOARD + VERDICT
# ================================================================================
if os.path.exists('outputs/delta_forecast/b2_comparison_scoreboard.csv'):
    df_b2 = pd.read_csv('outputs/delta_forecast/b2_comparison_scoreboard.csv')

    print('=' * 70)
    print('B2 — LEVEL vs DELTA: LOCKED TEST SCOREBOARD')
    print('=' * 70)

    # Core comparison columns
    cols = [
        'asset','horizon','b1_viability_tier','b1_delta_priority',
        'pers_MAE','pers_sMAPE','pers_DirAcc',
        'lvl_test_MAE','lvl_test_sMAPE','lvl_test_R2','lvl_test_DirAcc','lvl_test_MacroF1_2pct',
        'dlt_test_MAE','dlt_test_sMAPE','dlt_test_R2','dlt_test_DirAcc','dlt_test_MacroF1_2pct',
        'dlt_unc_resid_P10','dlt_unc_resid_P90','dlt_unc_interval_width_80pct',
        'win_score','win_verdict'
    ]
    display(df_b2[[c for c in cols if c in df_b2.columns]])

    print('\n--- VERDICT BREAKDOWN ---')
    print(df_b2['win_verdict'].value_counts().to_string())

    wins = df_b2[df_b2['win_verdict'].isin(['STRONG_WIN','WIN'])]
    total = len(df_b2)
    print(f'\nΔ-Ridge WIN or STRONG_WIN: {len(wins)}/{total} pairs')

    if len(wins) >= total * 0.6:
        print('-> Δ forecasting IMPROVES majority of pairs. B3 integration WARRANTED.')
    elif len(wins) >= total * 0.4:
        print('-> MIXED. Integrate winning pairs selectively. Do NOT blanket-upgrade.')
    else:
        print('-> KILL HYPOTHESIS. Δ does not beat naive. Attack features/model architecture.')

    print('\n--- TOP WINNING PAIRS ---')
    display(df_b2[df_b2['win_verdict'].isin(['STRONG_WIN','WIN'])][[
        'asset','horizon','dlt_test_sMAPE','dlt_test_R2','dlt_test_DirAcc',
        'dlt_test_MacroF1_2pct','dlt_unc_interval_width_80pct','win_verdict'
    ]])
else:
    print('Run Cell 3 first.')

In [ ]:
# ================================================================================
# CELL 6: DELTA vs LEVEL IMPROVEMENT TABLE
# (the one table that answers the key question)
# ================================================================================
if os.path.exists('outputs/delta_forecast/b2_comparison_scoreboard.csv'):
    df_b2 = pd.read_csv('outputs/delta_forecast/b2_comparison_scoreboard.csv')

    print('=' * 70)
    print('DID Δ FORECASTING ACTUALLY IMPROVE OVER LEVEL ON THE SAME LOCKED TEST SET?')
    print('=' * 70)
    print(f'{"Asset":>10} {"H":>4}  {"Tier":>10}  '
          f'{"MAE: Lvl→Δ":>14} {"sMAPE: Lvl→Δ":>14} '
          f'{"DA: Lvl→Δ":>13} {"F1@2%: Lvl→Δ":>14}  Verdict')
    print('-' * 110)

    for _, r in df_b2.sort_values(['asset','horizon']).iterrows():
        mae_chg   = r['dlt_test_MAE']          - r['lvl_test_MAE']
        smape_chg = r['dlt_test_sMAPE']        - r['lvl_test_sMAPE']
        da_chg    = r['dlt_test_DirAcc']       - r['lvl_test_DirAcc']
        f1_chg    = r['dlt_test_MacroF1_2pct'] - r['lvl_test_MacroF1_2pct']
        v = r['win_verdict']
        mk = '★' if v in ['STRONG_WIN','WIN'] else ' '
        print(f'{mk}{r["asset"].upper():>9} {r["horizon"]:>4}  {r["b1_viability_tier"]:>10}  '
              f'{mae_chg:>+13.0f} {smape_chg:>+13.2f}% '
              f'{da_chg:>+12.1f}% {f1_chg:>+13.1f}%  {v}')
else:
    print('Run Cell 3 first.')